In [1]:
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

In [2]:
# load 
df_all = pd.read_parquet('./STUV_0128_0514.parquet')

In [3]:
import matplotlib as mpl
from matplotlib import rcParams
## font manage
import matplotlib.font_manager as fm
font_path = '/gpfs01/euler/User/ydeng/Resource/Fonts/Arial.ttf'
font_prop = fm.FontProperties(fname=font_path)
fm.fontManager.addfont(font_path)
arial_font_name = font_prop.get_name()
print(f"Added font: {arial_font_name}")
rcParams['font.family'] = arial_font_name
rcParams['font.size'] = 9  # Common font size for all text
rcParams['svg.fonttype'] = 'none' # Ensure fonts are embedded as text, not converted to paths

Added font: Arial


In [4]:
rc_dict_boxplot = {'figure.dpi': 400,
                   'figure.figsize': [6, 4],
                   'figure.frameon': False,
                   'svg.fonttype':'none',
                   'font.family' :arial_font_name, 
    'font.size':9,
 'lines.markersize': 0.5,
 'lines.linewidth': .5,
 'lines.markeredgewidth': 0.5,
 'legend.columnspacing': 0.5,
 'legend.frameon': False,
 'legend.markerscale': 0.4,
 'legend.labelspacing': 0.1,
 'legend.handletextpad': 0,
 'legend.loc': 'lower right',
    'legend.fontsize': 9,
 'axes.labelpad': 1.0,
 'xtick.major.pad': 1,
 'ytick.major.pad': 1}

# Fig S2C (suppl.) for all recorded day

In [5]:
month_day_combos = df_all[['day', 'month']].drop_duplicates()

In [6]:
month_day_combos

,day,month
0,28,3
1055,14,5
2859,28,1
8734,29,3
10266,8,3
12081,19,4
18706,15,2
20090,12,2
21065,17,3
22149,14,2


# Fig 2C paper 14May for a single-day example

In [18]:
with mpl.rc_context(rc_dict_boxplot):
    chosen_day_months = [[14,5]]
    for ii in range(len(chosen_day_months)):
        fig,axes = plt.subplots(6,1,figsize=(4,4.2),sharex=True)
        day,month = chosen_day_months[ii]
        print(day,month)
        ## first column
        for tt,motor_tilt in enumerate([18,13,3,-7,-18]):
            ## AUC_M, AUC_S M_over_S
            
            
            df_= df_all[(df_all['day'] == day) & (df_all['month'] == month)& (df_all['motor_tilt'] == motor_tilt)]
            df_sorted = df_.sort_values(by='unix_ts', ascending=False)
            # seconds since midnight
            df_sorted['time_of_day_sec'] = (df_sorted['hour'] * 3600 +df_sorted['minute'] * 60 +df_sorted['second']
            )
            
            df_sorted['norm_int_auc_s'] = df_sorted['auc_scone']/df_sorted['int_t']
            df_sorted['norm_int_auc_m'] = df_sorted['auc_mcone']/df_sorted['int_t']
            axes[tt].plot(df_sorted['time_of_day_sec'],df_sorted['norm_int_auc_m'],color='green')
            axes[tt].plot(df_sorted['time_of_day_sec'],df_sorted['norm_int_auc_s'],color='blue')
            axes[tt].spines['top'].set_visible(False)
            axes[tt].set_yscale('log')
            axes[tt].set_ylim([1,1e5+1e3])
            
            axes[tt].set_yticks([1e2,1e4])
            axes[tt].set_yticklabels([r'$10^2$', r'$10^4$'])
            axes[2].set_ylabel('Photon count (a.u)')
            if tt!=0:
                axes[tt].tick_params(axis='y', labelleft=False)
            
            axes_ms = axes[tt].twinx()
            if tt==2:
                axes_ms.set_ylabel('M*/S*')
            if tt!=0:
                axes_ms.set_yticklabels([])
            
                
            
            axes_ms.spines['top'].set_visible(False)
            axes_ms.plot(df_sorted['time_of_day_sec'],df_sorted['auc_m_over_s'],color='k')

            axes[tt].set_xticks([ii*3600 for ii in [7,11,15,19]])
            axes_ms.set_ylim([0,30]);axes_ms.set_yticks([0,15,30]);axes_ms.set_yticklabels([0,15,30])
            if tt>0:
                axes_ms.set_yticklabels([])
            if tt<5:
                axes[tt].set_xticklabels([])
                axes_ms.set_xticklabels([])
                axes[tt].xaxis.set_visible(False);axes[tt].spines['bottom'].set_visible(False)
                axes_ms.xaxis.set_visible(False);axes_ms.spines['bottom'].set_visible(False)
        
        ## plot weather in the last row
        print(month,day)
        df_w = pd.read_csv(f"./weather_month{month}_day{day}.csv",skiprows=2)
        df_w.columns = [ "time","temperature_C","relative_humidity","precipitation","cloud_cover","wind_speed_kmh"]
        # convert time column
        df_w["time"] = pd.to_datetime(df_w["time"])
        df_w['time_of_day_sec'] = df_w['time'].dt.hour * 3600 +df_w['time'].dt.minute * 60 +df_w['time'].dt.second
       
        axes[-1].plot(df_w["time_of_day_sec"][7:22], df_w["precipitation"][7:22],color='lightblue',linewidth=2,label='rain (mm)')
        axes_cloud = axes[-1].twinx()
        axes_cloud.plot(df_w["time_of_day_sec"][7:22], df_w["cloud_cover"][7:22],color='gray',linewidth=2,label='cloud_cover(%)')
        axes[-1].set_xticks([ii*3600 for ii in [7,11,15,19]]);axes[-1].set_xticklabels(['7 a.m.','11 a.m.','3 p.m.','7 p.m.'], rotation = 30);axes[-1].set_xlim([7*3600-2400,21*3600+2400])
        axes_cloud.set_xticks([ii*3600 for ii in [7,11,15,19]]);axes_cloud.set_xticklabels(['7 a.m.','11 a.m.','3 p.m.','7 p.m.'], rotation = 30);axes_cloud.set_xlim([7*3600-2400,21*3600+2400])
        
        axes[-1].spines['top'].set_visible(False);axes[-1].set_ylabel('Rain (mm)')
        axes_cloud.spines['top'].set_visible(False);axes_cloud.set_ylabel('Cloud (%)')
        axes[-1].set_ylim([-.05,0.2])
        
        axes_cloud.set_ylim([0,100]);axes_cloud.set_yscale('linear');axes_cloud.set_xlim([7*3600-1000,21*3600+2400])
        
        fig.savefig(f'./Fig2C_v02062026.svg')
        #plt.close()
        #plt.show()

14 5
5 14


In [ ]:
df = pd.read_csv(
    f"weather_month{month}_day{day}.csv",
    skiprows=2
)

# rename columns for convenience
df.columns = [
    "time",
    "temperature_C",
    "relative_humidity",
    "precipitation",
    "cloud_cover",
    "wind_speed_kmh"
]

# convert time column
df["time"] = pd.to_datetime(df["time"])
df['time_of_day_sec'] = df['time'].dt.hour * 3600 +df['time'].dt.minute * 60 +df['time'].dt.second

# plot
fig, ax = plt.subplots(figsize=(8,4))

ax.plot(df["time_of_day_sec"], df["temperature_C"],color='red')
#ax.plot(df["time_of_day_sec"], df["precipitation"],color='blue')
ax.plot(df["time_of_day_sec"], df["cloud_cover"],color='gray')
ax.set_xticks([ii*3600 for ii in [7,11,15,19]]);ax.set_xticklabels(['7 a.m.','11 a.m.','3 p.m.','7 p.m.'], rotation = 30)
ax.set_xlabel("Time")
# ax.set_ylabel("Temperature (°C)")
# ax.set_title("Temperature over Time")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

ax.plot(df["time_of_day_sec"], df["temperature_C"])
ax.plot(df["time_of_day_sec"], df["precipitation_mm"])

ax.set_xlabel("Time")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature over Time")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()